In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\Moni\uncertainty-aware-rul-prediction")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

train_split = pd.read_csv(PROCESSED_DIR / "train_split.csv")
val_split = pd.read_csv(PROCESSED_DIR / "validation_split.csv")

print("Training columns:", train_split.columns.tolist())
print("Validation columns:", val_split.columns.tolist())

print("Training engines:", train_split["engine_id"].nunique())
print("Validation engines:", val_split["engine_id"].nunique())

print("Training rows ordered by engine and cycle:",
      train_split.sort_values(["engine_id", "cycle"]).index.equals(
          train_split.sort_values(["engine_id", "cycle"]).index
      ))

Training columns: ['engine_id', 'cycle', 'setting_1', 'setting_2', 'setting_3', 'sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21', 'RUL']
Validation columns: ['engine_id', 'cycle', 'setting_1', 'setting_2', 'setting_3', 'sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21', 'RUL']
Training engines: 174
Validation engines: 44
Training rows ordered by engine and cycle: True


In [2]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler

TARGET = "RUL"
ID_COLS = ["engine_id", "cycle"]
FEATURE_COLS = [
    col for col in train_split.columns
    if col not in ID_COLS + [TARGET]
]

WINDOW_SIZE = 30

# Fit scaler on training observations only
scaler = StandardScaler()
scaler.fit(train_split[FEATURE_COLS])

train_scaled = train_split.copy()
val_scaled = val_split.copy()

train_scaled[FEATURE_COLS] = scaler.transform(
    train_split[FEATURE_COLS]
)
val_scaled[FEATURE_COLS] = scaler.transform(
    val_split[FEATURE_COLS]
)


def create_windows(df, feature_cols, window_size):
    X, y = [], []

    for engine_id, engine_df in df.groupby("engine_id"):
        engine_df = engine_df.sort_values("cycle")

        features = engine_df[feature_cols].to_numpy(dtype=np.float32)
        rul = engine_df["RUL"].to_numpy(dtype=np.float32)

        # Only complete windows; never cross engine boundaries
        for end in range(window_size - 1, len(engine_df)):
            X.append(features[end - window_size + 1:end + 1])
            y.append(rul[end])

    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.float32)


X_train, y_train = create_windows(
    train_scaled, FEATURE_COLS, WINDOW_SIZE
)

X_val, y_val = create_windows(
    val_scaled, FEATURE_COLS, WINDOW_SIZE
)

print("Feature count:", len(FEATURE_COLS))
print("Window size:", WINDOW_SIZE)
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)
print("Training windows contain NaN:", np.isnan(X_train).any())
print("Validation windows contain NaN:", np.isnan(X_val).any())

Feature count: 24
Window size: 30
X_train shape: (31374, 30, 24)
y_train shape: (31374,)
X_val shape: (8222, 30, 24)
y_val shape: (8222,)
Training windows contain NaN: False
Validation windows contain NaN: False


In [5]:
%pip install torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import copy
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_absolute_error, mean_squared_error

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Convert arrays to tensors
train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32)
)
val_dataset = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.float32)
)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)


class RULCNN(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv1d(n_features, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        # (batch, time, features) -> (batch, features, time)
        return self.network(x.transpose(1, 2)).squeeze(1)


model = RULCNN(len(FEATURE_COLS)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

best_val_mae = float("inf")
best_state = None
patience = 5
epochs_without_improvement = 0
MAX_EPOCHS = 30

for epoch in range(MAX_EPOCHS):
    model.train()
    train_loss = 0.0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        predictions = model(xb)
        loss = criterion(predictions, yb)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * len(xb)

    model.eval()
    val_predictions = []
    val_targets = []

    with torch.no_grad():
        for xb, yb in val_loader:
            predictions = model(xb.to(device))
            val_predictions.extend(predictions.cpu().numpy())
            val_targets.extend(yb.numpy())

    val_mae = mean_absolute_error(val_targets, val_predictions)
    train_loss /= len(train_dataset)

    print(
        f"Epoch {epoch + 1:02d} | "
        f"Train MSE: {train_loss:.3f} | "
        f"Val MAE: {val_mae:.3f}"
    )

    if val_mae < best_val_mae:
        best_val_mae = val_mae
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
        print("Early stopping.")
        break

model.load_state_dict(best_state)

model.eval()
with torch.no_grad():
    X_val_tensor = torch.tensor(X_val, dtype=torch.float32).to(device)
    cnn_preds = model(X_val_tensor).cpu().numpy()

print("\nBest validation MAE:", round(best_val_mae, 3))
print("Final CNN RMSE:", round(
    mean_squared_error(y_val, cnn_preds) ** 0.5, 3
))

Device: cpu
  Using cached torch-2.14.1-cp314-cp314-win_amd64.whl.metadata (38 kB)
  Using cached filelock-4.2.4-py3-none-any.whl.metadata (2.0 kB)
  Using cached setuptools-84.2.1-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.7-py3-none-any.whl.metadata (6.7 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached torch-2.14.1-cp314-cp314-win_amd64.whl (124.1 MB)
Using cached fsspec-2026.9.0-py3-none-any.whl (221 kB)
Using cached networkx-3.7-py3-none-any.whl (2.1 MB)
Using cached setuptools-84.2.1-py3-none-any.whl (876 kB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
Using cached filelock-4.2.4-py3-none-any.whl (162 kB)

   ---------------------------------------- 0/7 [mpmath]
   ---------------------------------------- 0/7 [mpmath]
   ----------------------------


[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Epoch 01 | Train MSE: 6703.198 | Val MAE: 50.991
Epoch 02 | Train MSE: 3383.746 | Val MAE: 48.368
Epoch 03 | Train MSE: 2768.384 | Val MAE: 39.299
Epoch 04 | Train MSE: 1731.608 | Val MAE: 29.987
Epoch 05 | Train MSE: 1479.960 | Val MAE: 29.535
Epoch 06 | Train MSE: 1445.758 | Val MAE: 29.648
Epoch 07 | Train MSE: 1425.145 | Val MAE: 29.627
Epoch 08 | Train MSE: 1402.474 | Val MAE: 29.162
Epoch 09 | Train MSE: 1376.920 | Val MAE: 29.220
Epoch 10 | Train MSE: 1357.919 | Val MAE: 28.883
Epoch 11 | Train MSE: 1339.511 | Val MAE: 28.615
Epoch 12 | Train MSE: 1307.702 | Val MAE: 28.363
Epoch 13 | Train MSE: 1280.520 | Val MAE: 28.263
Epoch 14 | Train MSE: 1255.871 | Val MAE: 28.309
Epoch 15 | Train MSE: 1217.433 | Val MAE: 28.410
Epoch 16 | Train MSE: 1183.080 | Val MAE: 28.731
Epoch 17 | Train MSE: 1151.771 | Val MAE: 28.441
Epoch 18 | Train MSE: 1112.963 | Val MAE: 28.172
Epoch 19 | Train MSE: 1081.416 | Val MAE: 28.165
Epoch 20 | Train MSE: 1049.552 | Val MAE: 29.001
Epoch 21 | Train MSE

In [7]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

# Match the CNN's validation samples
val_eval = (
    val_split[val_split["cycle"] >= WINDOW_SIZE]
    .sort_values(["engine_id", "cycle"])
    .reset_index(drop=True)
)

features = FEATURE_COLS

rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(train_split[features], train_split["RUL"])
rf_preds = rf_model.predict(val_eval[features])

# CNN predictions already follow engine/cycle window order
cnn_preds = cnn_preds.reshape(-1)

assert len(cnn_preds) == len(val_eval), (
    f"CNN predictions: {len(cnn_preds)}, "
    f"matching validation rows: {len(val_eval)}"
)

comparison = pd.DataFrame([
    {
        "Model": "Random Forest",
        "MAE": mean_absolute_error(val_eval["RUL"], rf_preds),
        "RMSE": np.sqrt(mean_squared_error(val_eval["RUL"], rf_preds))
    },
    {
        "Model": "PyTorch 1D CNN",
        "MAE": mean_absolute_error(val_eval["RUL"], cnn_preds),
        "RMSE": np.sqrt(mean_squared_error(val_eval["RUL"], cnn_preds))
    }
])

print(comparison.round(3).to_string(index=False))

         Model    MAE   RMSE
 Random Forest 26.853 35.655
PyTorch 1D CNN 28.165 37.936


In [9]:
import json
from pathlib import Path
import joblib
import torch

PROJECT_ROOT = Path(
    r"C:\Users\Moni\uncertainty-aware-rul-prediction"
)

MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save the best CNN weights
torch.save(
    model.state_dict(),
    MODEL_DIR / "rul_cnn.pt"
)

# Save the fitted scaler and feature configuration
joblib.dump(
    scaler,
    MODEL_DIR / "feature_scaler.joblib"
)

with open(MODEL_DIR / "model_config.json", "w") as f:
    json.dump({
        "model_type": "RULCNN",
        "window_size": WINDOW_SIZE,
        "feature_columns": FEATURE_COLS,
        "target": "RUL",
        "best_validation_mae": float(best_val_mae)
    }, f, indent=4)

print("Saved CNN, scaler, and model configuration to:", MODEL_DIR)

Saved CNN, scaler, and model configuration to: C:\Users\Moni\uncertainty-aware-rul-prediction\models


In [10]:
from pathlib import Path

MODEL_DIR = Path(
    r"C:\Users\Moni\uncertainty-aware-rul-prediction\models"
)

expected_files = [
    "rul_cnn.pt",
    "feature_scaler.joblib",
    "model_config.json"
]

for filename in expected_files:
    path = MODEL_DIR / filename
    print(f"{filename}: {'OK' if path.exists() else 'MISSING'}")

rul_cnn.pt: OK
feature_scaler.joblib: OK
model_config.json: OK
